In [2]:
import pandas as pd
import numpy as np
import os
import re
from sqlalchemy import create_engine, inspect, text
from IPython.display import Image, display
from collections import defaultdict

In [17]:
# Ruta donde están los archivos CSV
FOLDER_PATH = r'C:\Users\Asus\Documents\IT\Especialidad_IT_Academy\Proyecto\ddbb'

# Lista oficial de las 24 jurisdicciones (id_normalizado, nombre_completo)
JURISDICCIONES_OFICIALES = [
    ('buenos aires', 'buenos aires'),
    ('caba', 'ciudad autónoma de buenos aires'),
    ('catamarca', 'catamarca'),
    ('chaco', 'chaco'),
    ('chubut', 'chubut'),
    ('córdoba', 'córdoba'),
    ('corrientes', 'corrientes'),
    ('entre ríos', 'entre ríos'),
    ('formosa', 'formosa'),
    ('jujuy', 'jujuy'),
    ('la pampa', 'la pampa'),
    ('la rioja', 'la rioja'),
    ('mendoza', 'mendoza'),
    ('misiones', 'misiones'),
    ('neuquén', 'neuquén'),
    ('río negro', 'río negro'),
    ('salta', 'salta'),
    ('san juan', 'san juan'),
    ('san luis', 'san luis'),
    ('santa cruz', 'santa cruz'),
    ('santa fe', 'santa fe'),
    ('santiago del estero', 'santiago del estero'),
    ('tierra del fuego', 'tierra del fuego, antártida e islas del atlántico sur'),
    ('tucumán', 'tucumán')
]

# Personas trans identificadas manualmente
PERSONAS_TRANS_RAW = {
    ('ibañez', 'reina xiomara'),
    ('antunez', 'julieta'),
    ('infante', 'ornela reina')
}

# Clasificación ideológica de partidos (1=progresista, 5=conservador) - PENDIENTE!  
CLASIFICACION_PARTIDOS = {
    'frente de izquierda': {'espacio': 'FIT', 'eje': 1},
    'partido obrero': {'espacio': 'FIT', 'eje': 1},
    'mst': {'espacio': 'FIT', 'eje': 1},
    'izquierda': {'espacio': 'FIT', 'eje': 1},
    'union por la patria': {'espacio': 'UP', 'eje': 2},
    'frente de todos': {'espacio': 'UP', 'eje': 2},
    'partido justicialista': {'espacio': 'UP', 'eje': 2},
    'justicialista': {'espacio': 'UP', 'eje': 2},
    'coalicion civica': {'espacio': 'Centro', 'eje': 3},
    'partido socialista': {'espacio': 'Centro', 'eje': 3},
    'gen': {'espacio': 'Centro', 'eje': 3},
    'juntos por el cambio': {'espacio': 'JxC', 'eje': 4},
    'pro': {'espacio': 'JxC', 'eje': 4},
    'union civica radical': {'espacio': 'JxC', 'eje': 4},
    'ucr': {'espacio': 'JxC', 'eje': 4},
    'la libertad avanza': {'espacio': 'LLA', 'eje': 5},
    'partido democrata': {'espacio': 'LLA', 'eje': 5},
    'libertad avanza': {'espacio': 'LLA', 'eje': 5},
}

# Unificación de nombres de partidos (evita duplicados)
UNIFICAR_PARTIDOS = {
    'pro': 'propuesta republicana',
    'ucr': 'union civica radical',
    'cambiemos': 'juntos por el cambio',
    'frente de todos': 'union por la patria',
    'libertad avanza': 'la libertad avanza',
}

# Unificación de nombres de personas (casos detectados manualmente)
UNIFICAR_PERSONAS = {
    ('fernandez', 'elia marina'): ('fernandez de mansilla', 'elia'),
    ('brugge', 'eliana'): ('bruno', 'eliana'),
    ('ali', 'ernesto pipi'): ('ali', 'ernesto'),
}

In [18]:
# Definición de funciones auxiliares

def normalizar_texto(texto):
    """Convierte texto a minúsculas, sin tildes ni caracteres especiales."""
    if not texto or pd.isna(texto):
        return None
    texto = str(texto).lower().strip()
    texto = texto.replace('á', 'a').replace('é', 'e').replace('í', 'i')
    texto = texto.replace('ó', 'o').replace('ú', 'u').replace('ñ', 'n')
    texto = re.sub(r'[^a-z0-9_\-\s]', '', texto)
    return texto


def normalizar_genero(genero):
    """Clasifica el género en categorías estandarizadas."""
    if not genero or pd.isna(genero):
        return 'no_informado'
    # Limpiar: minúsculas, strip (quita espacios adelante/atrás)
    g = str(genero).lower().strip()
    # Femenino
    if g in ['femenino', 'mujer', 'f']:
        return 'femenino'
    # Masculino (incluye 'n' como masculino)
    if g in ['masculino', 'varon', 'hombre', 'h', 'n']:
        return 'masculino'
    # No binario
    if g in ['no binario', 'no binarie', 'nb']:
        return 'no_binario'
    # Casos especiales de falta de dato
    if g in ['', 's/d', 'sin datos', 'no informado']:
        return 'no_informado'
    # Por defecto
    return 'no_informado'


def normalizar_jurisdiccion(nombre):
    """Estandariza nombres de provincias y CABA."""
    if not nombre:
        return None
    nombre = normalizar_texto(nombre)
    if not nombre:
        return None
    # Mapeo de variantes comunes
    if nombre in ['caba', 'ciudad autonoma de buenos aires', 'ciudad de buenos aires']:
        return 'caba'
    if nombre in ['cordoba', 'cba']:
        return 'córdoba'
    if nombre == 'tucuman':
        return 'tucumán'
    if nombre == 'neuquen':
        return 'neuquén'
    if nombre in ['rionegro', 'rio negro']:
        return 'río negro'
    if nombre in ['entrerios', 'entre rios']:
        return 'entre ríos'
    if nombre in ['tierra del fuego aias', 'tierra del fuego']:
        return 'tierra del fuego'
    return nombre


def identificar_personas_trans():
    """Normaliza los nombres de las personas trans para su identificación."""
    personas_trans = set()
    for apellido, nombre in PERSONAS_TRANS_RAW:
        ap_clean = normalizar_texto(apellido)
        nom_clean = normalizar_texto(nombre)
        if ap_clean and nom_clean:
            personas_trans.add((ap_clean, nom_clean))
    return personas_trans

In [19]:
# Conectar base de datos SQLite

engine = create_engine('sqlite:///modelo_limpio.db', echo=False)
print("Base de datos 'modelo_limpio.db' lista")

Base de datos 'modelo_limpio.db' lista


In [20]:
# Cargar todos los CSVs en un diccionario

print("Cargando archivos CSV...")

# Buscar todos los archivos .csv dentro de la carpeta
csv_files = [f for f in os.listdir(FOLDER_PATH) if f.endswith('.csv')]
print(f"Archivos encontrados: {len(csv_files)}")

# Crear un diccionario vacío para guardar los DataFrames
dataframes = {}

# Recorrer cada archivo y cargarlo
for file in csv_files:
    file_path = os.path.join(FOLDER_PATH, file)
    df_name = os.path.splitext(file)[0]
    
    try:
        dataframes[df_name] = pd.read_csv(file_path)
    except Exception as e:
        print(f"No se pudo cargar {file}: {e}")

print(f"Se cargaron {len(dataframes)} DataFrames")

Cargando archivos CSV...
Archivos encontrados: 64
Se cargaron 64 DataFrames


In [21]:
# Ruta de la carpeta
folder_path = r'C:\Users\Asus\Documents\IT\Especialidad_IT_Academy\Proyecto\ddbb'

# Diccionario para agrupar CSVs por estructura de columnas
estructuras = defaultdict(list)

print("📂 Explorando archivos CSV...")
print("="*60)

for file in os.listdir(folder_path):
    if file.endswith('.csv'):
        file_path = os.path.join(folder_path, file)
        try:
            df = pd.read_csv(file_path, nrows=0)  # solo lee encabezados
            columnas = tuple(df.columns)
            estructuras[columnas].append(file)
        except Exception as e:
            print(f"❌ Error leyendo {file}: {e}")

# Mostrar resultados agrupados
for columnas, archivos in estructuras.items():
    print(f"\n📋 ESTRUCTURA con {len(columnas)} columnas")
    print(f"   Columnas: {list(columnas)}")
    print(f"   Cantidad de archivos: {len(archivos)}")
    print(f"   Ejemplos:")
    for a in archivos[:3]:
        print(f"      - {a}")

📂 Explorando archivos CSV...

📋 ESTRUCTURA con 8 columnas
   Columnas: ['region', 'jurisdiccion', 'diputadxs', 'cantidad diputadas', 'porcentaje diputadas', 'senadorxs', 'cantidad senadoras', 'porcentaje senadoras']
   Cantidad de archivos: 3
   Ejemplos:
      - 1. Congreso Nacional 2021-2023 - Base.csv
      - 15. Congreso Nacional 2025-2027 - Base.csv
      - 9. Congreso Nacional 2023-2025 - Base.csv

📋 ESTRUCTURA con 7 columnas
   Columnas: ['composicion', 'mandato', 'apellido', 'nombre', 'genero', 'provincia', 'bloque']
   Cantidad de archivos: 2
   Ejemplos:
      - 1. Congreso Nacional 2021-2023 - HCDN.csv
      - 1. Congreso Nacional 2021-2023 - HSN.csv

📋 ESTRUCTURA con 11 columnas
   Columnas: ['distrito', 'listas', 'listas encabezadas por mujeres', '% listas encabezadas por mujeres', 'listas competitivas', 'listas competitivas encabezadas por mujeres', '% listas competitivas encabezadas por mujeres', 'LCEM peronismo', 'LCEM la libertad avanza + pro', 'LCEM izquierda', 'LCEM 

### Transformaciones

#### CN_base

In [22]:
# Agregar columna 'composicion' con valor predeterminado para cada df Base
dataframes['1. Congreso Nacional 2021-2023 - Base']['composicion'] = '2021-2023'
dataframes['9. Congreso Nacional 2023-2025 - Base']['composicion'] = '2023-2025'
dataframes['15. Congreso Nacional 2025-2027 - Base']['composicion'] = '2025-2027'

In [23]:
# Unir los tres df en uno solo
CN_base = pd.concat([
    dataframes['1. Congreso Nacional 2021-2023 - Base'],
    dataframes['9. Congreso Nacional 2023-2025 - Base'],
    dataframes['15. Congreso Nacional 2025-2027 - Base']
], ignore_index=True)

In [24]:
CN_base

,region,jurisdiccion,diputadxs,cantidad diputadas,porcentaje diputadas,senadorxs,cantidad senadoras,porcentaje senadoras,composicion
0,buenos aires,buenos aires,70,34,49%,3,2,67%,2021-2023
1,noroeste argentino,catamarca,5,2,40%,3,1,33%,2021-2023
2,noreste argentino,chaco,7,3,43%,3,1,33%,2021-2023
3,patagonia,chubut,5,3,60%,3,1,33%,2021-2023
4,ciudad autonoma de buenos aires,ciudad autonoma de buenos aires,25,13,52%,3,1,33%,2021-2023
...,...,...,...,...,...,...,...,...,...
70,centro,santa fe,19,9,47%,3,1,33%,2025-2027
71,noroeste argentino,santiago del estero,7,4,57%,3,1,33%,2025-2027
72,patagonia,tierra del fuego aias,5,1,20%,3,2,67%,2025-2027
73,noroeste argentino,tucuman,9,3,33%,3,2,67%,2025-2027


In [25]:
CN_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   region                75 non-null     str  
 1   jurisdiccion          73 non-null     str  
 2   diputadxs             75 non-null     int64
 3   cantidad diputadas    75 non-null     int64
 4   porcentaje diputadas  75 non-null     str  
 5   senadorxs             75 non-null     int64
 6   cantidad senadoras    75 non-null     int64
 7   porcentaje senadoras  75 non-null     str  
 8   composicion           75 non-null     str  
dtypes: int64(4), str(5)
memory usage: 5.4 KB


In [26]:
# Filtrar filas donde una columna específica es nula
CN_base[CN_base['jurisdiccion'].isna()]

,region,jurisdiccion,diputadxs,cantidad diputadas,porcentaje diputadas,senadorxs,cantidad senadoras,porcentaje senadoras,composicion
49,total pais,NaN,257,109,42%,72,34,47%,2023-2025
74,total pais,NaN,257,106,41%,72,33,46%,2025-2027


In [27]:
# Quitar el '%' del valor y convertir a float
CN_base['porcentaje diputadas'] = CN_base['porcentaje diputadas'].str.replace('%', '').astype(float)
CN_base['porcentaje senadoras'] = CN_base['porcentaje senadoras'].str.replace('%', '').astype(float)

In [28]:
# normalizar jurisdiccion
CN_base['jurisdiccion'] = CN_base['jurisdiccion'].apply(normalizar_jurisdiccion)

In [29]:
CN_base

,region,jurisdiccion,diputadxs,cantidad diputadas,porcentaje diputadas,senadorxs,cantidad senadoras,porcentaje senadoras,composicion
0,buenos aires,buenos aires,70,34,49.0,3,2,67.0,2021-2023
1,noroeste argentino,catamarca,5,2,40.0,3,1,33.0,2021-2023
2,noreste argentino,chaco,7,3,43.0,3,1,33.0,2021-2023
3,patagonia,chubut,5,3,60.0,3,1,33.0,2021-2023
4,ciudad autonoma de buenos aires,caba,25,13,52.0,3,1,33.0,2021-2023
...,...,...,...,...,...,...,...,...,...
70,centro,santa fe,19,9,47.0,3,1,33.0,2025-2027
71,noroeste argentino,santiago del estero,7,4,57.0,3,1,33.0,2025-2027
72,patagonia,tierra del fuego,5,1,20.0,3,2,67.0,2025-2027
73,noroeste argentino,tucumán,9,3,33.0,3,2,67.0,2025-2027


In [30]:
CN_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   region                75 non-null     str    
 1   jurisdiccion          73 non-null     str    
 2   diputadxs             75 non-null     int64  
 3   cantidad diputadas    75 non-null     int64  
 4   porcentaje diputadas  75 non-null     float64
 5   senadorxs             75 non-null     int64  
 6   cantidad senadoras    75 non-null     int64  
 7   porcentaje senadoras  75 non-null     float64
 8   composicion           75 non-null     str    
dtypes: float64(2), int64(4), str(3)
memory usage: 5.4 KB


#### HDCN y HSN

In [31]:
# Renombrar columna 'provincia' a 'jurisdiccion' en los df del CN 2021-2023 de diputados y senado (para que coincida con los de los siguientes períodos)
dataframes['1. Congreso Nacional 2021-2023 - HCDN'].rename(columns={'provincia': 'jurisdiccion'}, inplace=True)
dataframes['1. Congreso Nacional 2021-2023 - HSN'].rename(columns={'provincia': 'jurisdiccion'}, inplace=True)

In [32]:
# Corregir columna 'composicion' en el df de diputados 2025-2027 (algunas rows pone 2023-2027)
dataframes['15. Congreso Nacional 2025-2027 - HCDN']['composicion'] = '2025-2027'

In [33]:
# Unir los df de diputados por un lado y los de senado por otro
CN_HCDN = pd.concat([
    dataframes['1. Congreso Nacional 2021-2023 - HCDN'],
    dataframes['9. Congreso Nacional 2023-2025 - HCDN'],
    dataframes['15. Congreso Nacional 2025-2027 - HCDN']
], ignore_index=True)

CN_HSN = pd.concat([
    dataframes['1. Congreso Nacional 2021-2023 - HSN'],
    dataframes['9. Congreso Nacional 2023-2025 - HSN'],
    dataframes['15. Congreso Nacional 2025-2027 - HSN']
], ignore_index=True)

In [34]:
CN_HCDN.describe()

,composicion,mandato,apellido,nombre,genero,jurisdiccion,bloque,jurisdicción
count,1028,1028,1028,1028,1028,514,1028,514
unique,4,5,555,479,4,25,60,25
top,2019-2021,2023-2027,Fernandez,Pablo,masculino,buenos aires,frente de todos,buenos aires
freq,257,259,9,13,585,140,237,139


In [35]:
# Eliminar columna 'jurisdicción'
CN_HCDN.drop(columns=['jurisdicción'], inplace=True)

In [36]:
CN_HCDN['genero'].unique()

<StringArray>
['femenino', 'masculino', 'N', 'masculino ']
Length: 4, dtype: str

In [37]:
CN_HCDN[CN_HCDN['genero'].str.strip() == 'N']
# es masculino

,composicion,mandato,apellido,nombre,genero,jurisdiccion,bloque
420,2021-2023,2021-2025,Monzó,Emilio,N,buenos aires,encuentro federal


In [38]:
# Normalizar género
CN_HCDN['genero'] = CN_HCDN['genero'].apply(normalizar_genero)

In [39]:
CN_HCDN.describe()

,composicion,mandato,apellido,nombre,genero,jurisdiccion,bloque
count,1028,1028,1028,1028,1028,514,1028
unique,4,5,555,479,2,25,60
top,2019-2021,2023-2027,Fernandez,Pablo,masculino,buenos aires,frente de todos
freq,257,259,9,13,587,140,237


In [40]:
# normalizar jurisdiccion
CN_HCDN['jurisdiccion'] = CN_HCDN['jurisdiccion'].apply(normalizar_jurisdiccion)

In [41]:
# normalizar columnas apellido y nombre
CN_HCDN['apellido'] = CN_HCDN['apellido'].apply(normalizar_texto)
CN_HCDN['nombre'] = CN_HCDN['nombre'].apply(normalizar_texto)

In [42]:
CN_HCDN

,composicion,mandato,apellido,nombre,genero,jurisdiccion,bloque
0,2019-2021,2017-2021,abdala de matarazzo,norma amanda,femenino,santiago del estero,frente de todos
1,2019-2021,2019-2023,aguirre,hilda clelia,femenino,la rioja,frente de todos
2,2019-2021,2017-2021,aicega,juan,masculino,buenos aires,pro
3,2019-2021,2019-2023,alderete,juan carlos,masculino,buenos aires,frente de todos
4,2019-2021,2017-2021,allende,walberto,masculino,san juan,frente de todos
...,...,...,...,...,...,...,...
1023,2025-2027,2023-2027,zago,oscar roberto,masculino,NaN,mid movimiento de integración y desarrollo
1024,2025-2027,2025-2029,zapata,carlos saul,masculino,NaN,la libertad avanza
1025,2025-2027,2023-2027,zaracho,natalia,femenino,NaN,fuerza patria
1026,2025-2027,2025-2029,zigaran,maria ines,femenino,NaN,provincias unidas


In [43]:
CN_HSN.describe()

,composicion,mandato,apellido,nombre,genero,jurisdiccion,bloque,jurisdicción
count,288,288,288,288,288,144,288,144
unique,5,6,141,140,2,24,30,25
top,2019-2021,2019-2025,González,Martín,masculino,corrientes,union civica radical,salta
freq,72,72,7,6,161,6,46,6


In [44]:
CN_HSN['composicion'].unique()

<StringArray>
['2019-2021', '2021-2023', '2023-2025', '2023-2026', '2025-2027']
Length: 5, dtype: str

In [45]:
CN_HSN[CN_HSN['composicion'].str.strip() == '2023-2026']

,composicion,mandato,apellido,nombre,genero,jurisdiccion,bloque,jurisdicción
150,2023-2026,2021-2027,Corpacci,Lucia Benigna,femenino,NaN,frente nacional y popular,catamarca


In [46]:
# Corregir '2023-2026' a '2023-2025' en esa row
CN_HSN.loc[150, 'composicion'] = '2023-2025'

In [47]:
CN_HSN['composicion'].unique()

<StringArray>
['2019-2021', '2021-2023', '2023-2025', '2025-2027']
Length: 4, dtype: str

In [48]:
CN_HSN['genero'].unique()

<StringArray>
['femenino', 'masculino']
Length: 2, dtype: str

In [49]:
# normalizar jurisdiccion, nombre y apellido
CN_HSN['jurisdiccion'] = CN_HSN['jurisdiccion'].apply(normalizar_jurisdiccion)
CN_HSN['apellido'] = CN_HSN['apellido'].apply(normalizar_texto)
CN_HSN['nombre'] = CN_HSN['nombre'].apply(normalizar_texto)

In [50]:
CN_HSN

,composicion,mandato,apellido,nombre,genero,jurisdiccion,bloque,jurisdicción
0,2019-2021,2015-2021,almiron,ana claudia,femenino,corrientes,frente de todos,NaN
1,2019-2021,2015-2021,alperovich,jose jorge,masculino,tucumán,frente de todos,NaN
2,2019-2021,2017-2023,basualdo,roberto gustavo,masculino,san juan,producción y trabajo,NaN
3,2019-2021,2019-2025,blanco,pablo daniel,masculino,tierra del fuego,union civica radical,NaN
4,2019-2021,2015-2021,blas,ines imelda,femenino,catamarca,frente de todos,NaN
...,...,...,...,...,...,...,...,...
283,2025-2027,2023-2029,unac,sergio mario,masculino,NaN,justicialista,san juan
284,2025-2027,2021-2027,valenzuela,mercedes gabriela,femenino,NaN,unión cívica radical,corrientes
285,2025-2027,2021-2027,vigo,alejandra maria,femenino,NaN,provincias unidas,córdoba
286,2025-2027,2021-2027,vischi,eduardo alejandro,masculino,NaN,unión cívica radical,corrientes


#### 25-27_legislativos

In [51]:
# filtrar solo Nación
CN_25_27 = dataframes['25-27_legislativos - base']
CN_25_27['tipo'].unique()

<StringArray>
['legislatura', 'congreso nac']
Length: 2, dtype: str

In [52]:
CN_25_27 = CN_25_27[CN_25_27['tipo'] == 'congreso nac']
CN_25_27

,tipo,camara,provincia,mandato,apellido,nombre,genero,seccion,bloque
1198,congreso nac,diputados,na,2025-2029,Aguero,Guillermo César,masculino,chaco,unión cívica radical
1199,congreso nac,diputados,na,2023-2027,Aguirre,Hilda Celia,femenino,la rioja,fuerza patria
1200,congreso nac,diputados,na,2025-2029,Ajmechet,Sabrina Carlota,femenino,ciudad de buenos aires,la libertad avanza
1201,congreso nac,diputados,na,2023-2027,Ali,Ernesto,masculino,san luis,fuerza patria
1202,congreso nac,diputados,na,2025-2029,Almena,Carlos Alberto,masculino,san luis,la libertad avanza
...,...,...,...,...,...,...,...,...,...
1522,congreso nac,senado,na,2023-2029,Uñac,Sergio Mario,masculino,san juan,justicialista
1523,congreso nac,senado,na,2021-2027,Valenzuela,Mercedes Gabriela,femenino,corrientes,unión cívica radical
1524,congreso nac,senado,na,2021-2027,Vigo,Alejandra María,femenino,córdoba,provincias unidas
1525,congreso nac,senado,na,2021-2027,Vischi,Eduardo Alejandro,masculino,corrientes,unión cívica radical


In [53]:
# eliminar columnas tipo y provincia (no aplican)
CN_25_27.drop(columns=['tipo', 'provincia'], inplace=True)

In [54]:
# renombrar 'seccion' por 'jurisdiccion'
CN_25_27.rename(columns={'seccion':'jurisdiccion'}, inplace=True)

In [55]:
# normalizar jurisdiccion, nombre y apellido
CN_25_27['jurisdiccion'] = CN_25_27['jurisdiccion'].apply(normalizar_jurisdiccion)
CN_25_27['apellido'] = CN_25_27['apellido'].apply(normalizar_texto)
CN_25_27['nombre'] = CN_25_27['nombre'].apply(normalizar_texto)

In [56]:
CN_25_27

,camara,mandato,apellido,nombre,genero,jurisdiccion,bloque
1198,diputados,2025-2029,aguero,guillermo cesar,masculino,chaco,unión cívica radical
1199,diputados,2023-2027,aguirre,hilda celia,femenino,la rioja,fuerza patria
1200,diputados,2025-2029,ajmechet,sabrina carlota,femenino,caba,la libertad avanza
1201,diputados,2023-2027,ali,ernesto,masculino,san luis,fuerza patria
1202,diputados,2025-2029,almena,carlos alberto,masculino,san luis,la libertad avanza
...,...,...,...,...,...,...,...
1522,senado,2023-2029,unac,sergio mario,masculino,san juan,justicialista
1523,senado,2021-2027,valenzuela,mercedes gabriela,femenino,corrientes,unión cívica radical
1524,senado,2021-2027,vigo,alejandra maria,femenino,córdoba,provincias unidas
1525,senado,2021-2027,vischi,eduardo alejandro,masculino,corrientes,unión cívica radical


#### Encabezamientos 2023

- csv 5. (del año 2023, en donde hubo elecciones generales y PASO)

In [57]:
E_23_PASO_HCDN = dataframes['5. Encabezamientos legislativos nacionales 2023 - Base PASO HCDN']
E_23_PASO_HSN = dataframes['5. Encabezamientos legislativos nacionales 2023 - Base PASO HSN']
E_23_generales_HCDN = dataframes['5. Encabezamientos legislativos nacionales 2023 - Base generales HCDN']
E_23_generales_HSN = dataframes['5. Encabezamientos legislativos nacionales 2023 - Base generales HSN']

In [58]:
# Renombrar la columna 'distrito' a 'jurisdiccion y normalizar
E_23_PASO_HCDN.rename(columns={'distrito':'jurisdiccion'}, inplace=True)
E_23_PASO_HCDN['jurisdiccion'] = E_23_PASO_HCDN['jurisdiccion'].apply(normalizar_jurisdiccion)
E_23_PASO_HSN.rename(columns={'distrito':'jurisdiccion'}, inplace=True)
E_23_PASO_HSN['jurisdiccion'] = E_23_PASO_HSN['jurisdiccion'].apply(normalizar_jurisdiccion)
E_23_generales_HCDN.rename(columns={'distrito':'jurisdiccion'}, inplace=True)
E_23_generales_HCDN['jurisdiccion'] = E_23_generales_HCDN['jurisdiccion'].apply(normalizar_jurisdiccion)
E_23_generales_HSN.rename(columns={'distrito':'jurisdiccion'}, inplace=True)
E_23_generales_HSN['jurisdiccion'] = E_23_generales_HSN['jurisdiccion'].apply(normalizar_jurisdiccion)

In [59]:
# Renombrar la columna 'listas encabezadas por mujeres sobre el total de listas' a '% listas encabezadas por mujeres'
E_23_PASO_HCDN.rename(columns={'listas encabezadas por mujeres sobre el total de listas':'% listas encabezadas por mujeres'}, inplace=True)
E_23_PASO_HSN.rename(columns={'listas encabezadas por mujeres sobre el total de listas':'% listas encabezadas por mujeres'}, inplace=True)
E_23_generales_HCDN.rename(columns={'listas encabezadas por mujeres sobre el total de listas':'% listas encabezadas por mujeres'}, inplace=True)
E_23_generales_HSN.rename(columns={'listas encabezadas por mujeres sobre el total de listas':'% listas encabezadas por mujeres'}, inplace=True)

In [60]:
# Renombrar la columna 'porcentaje listas competitivas encabezadas por mujeres sobre el total de listas competitivas' a '% listas competitivas encabezadas por mujeres'
E_23_PASO_HCDN.rename(columns={'porcentaje listas competitivas encabezadas por mujeres sobre el total de listas competitivas':'% listas competitivas encabezadas por mujeres'}, inplace=True)
E_23_PASO_HSN.rename(columns={'porcentaje listas competitivas encabezadas por mujeres sobre el total de listas competitivas':'% listas competitivas encabezadas por mujeres'}, inplace=True)
E_23_generales_HCDN.rename(columns={'porcentaje listas competitivas encabezadas por mujeres sobre el total de listas competitivas':'% listas competitivas encabezadas por mujeres'}, inplace=True)
E_23_generales_HSN.rename(columns={'porcentaje listas competitivas encabezadas por mujeres sobre el total de listas competitivas':'% listas competitivas encabezadas por mujeres'}, inplace=True)

In [61]:
# Quitar el '%' del valor de las columnas '% listas encabezadas por mujeres' y '% listas competitivas encabezadas por mujeres', y convertir a float
E_23_PASO_HCDN['% listas encabezadas por mujeres'] = E_23_PASO_HCDN['% listas encabezadas por mujeres'].str.replace('%', '').astype(float)
E_23_PASO_HCDN['% listas competitivas encabezadas por mujeres'] = E_23_PASO_HCDN['% listas competitivas encabezadas por mujeres'].str.replace('%', '').astype(float)
E_23_PASO_HSN['% listas encabezadas por mujeres'] = E_23_PASO_HSN['% listas encabezadas por mujeres'].str.replace('%', '').astype(float)
E_23_PASO_HSN['% listas competitivas encabezadas por mujeres'] = E_23_PASO_HSN['% listas competitivas encabezadas por mujeres'].str.replace('%', '').astype(float)
E_23_generales_HCDN['% listas encabezadas por mujeres'] = E_23_generales_HCDN['% listas encabezadas por mujeres'].str.replace('%', '').astype(float)
E_23_generales_HCDN['% listas competitivas encabezadas por mujeres'] = E_23_generales_HCDN['% listas competitivas encabezadas por mujeres'].str.replace('%', '').astype(float)
E_23_generales_HSN['% listas encabezadas por mujeres'] = E_23_generales_HSN['% listas encabezadas por mujeres'].str.replace('%', '').astype(float)
E_23_generales_HSN['% listas competitivas encabezadas por mujeres'] = E_23_generales_HSN['% listas competitivas encabezadas por mujeres'].str.replace('%', '').astype(float)

In [62]:
# Renombrar columnas 'listas competitivas encabezadas por mujeres.1' a 'LCEM unión por la patria'
E_23_PASO_HCDN.rename(columns={'listas competitivas encabezadas por mujeres.1':'LCEM unión por la patria'}, inplace=True)
E_23_PASO_HSN.rename(columns={'listas competitivas encabezadas por mujeres.1':'LCEM unión por la patria'}, inplace=True)
E_23_generales_HCDN.rename(columns={'listas competitivas encabezadas por mujeres.1':'LCEM unión por la patria'}, inplace=True)

In [63]:
# Agregar columna 'camara' para luego unir los df por tipo de elección (según se trate de las PASO o las generales)
E_23_PASO_HCDN['camara'] = 'HCDN'
E_23_generales_HCDN['camara'] = 'HCDN'
E_23_PASO_HSN['camara'] = 'HSN'
E_23_generales_HSN['camara'] = 'HSN'

In [64]:
# Unir los df por tipo de elección (según se trate de las PASO o las generales)
# PASO
E_23_PASO = pd.concat([
    E_23_PASO_HCDN,
    E_23_PASO_HSN]
    , ignore_index=True)
# Generales
E_23_generales = pd.concat([
    E_23_generales_HCDN,
    E_23_generales_HSN]
    , ignore_index=True)

In [65]:
E_23_PASO

,jurisdiccion,listas,listas encabezadas por mujeres,% listas encabezadas por mujeres,listas competitivas,listas competitivas encabezadas por mujeres,% listas competitivas encabezadas por mujeres,LCEM unión por la patria,LCEM juntos por el cambio,LCEM la libertad avanza,LCEM izquierda,LCEM alianza/partido provincial,camara
0,buenos aires,30,9,30.0,5,1,20.0,NaN,NaN,NaN,NaN,1.0,HCDN
1,caba,21,6,29.0,7,3,43.0,1.0,NaN,1.0,1.0,NaN,HCDN
2,catamarca,13,5,38.0,2,0,0.0,NaN,NaN,NaN,NaN,NaN,HCDN
3,chaco,9,3,33.0,2,0,0.0,NaN,NaN,NaN,NaN,NaN,HCDN
4,chubut,8,1,13.0,5,1,20.0,1.0,NaN,NaN,NaN,NaN,HCDN
5,córdoba,27,8,30.0,4,1,25.0,1.0,NaN,NaN,NaN,NaN,HCDN
6,corrientes,9,3,33.0,5,1,20.0,1.0,NaN,NaN,NaN,NaN,HCDN
7,entre ríos,5,0,0.0,3,0,0.0,NaN,NaN,NaN,NaN,NaN,HCDN
8,formosa,6,3,50.0,3,2,67.0,1.0,1.0,NaN,NaN,NaN,HCDN
9,jujuy,11,4,36.0,8,4,50.0,NaN,2.0,NaN,2.0,NaN,HCDN


In [66]:
E_23_generales

,jurisdiccion,listas,listas encabezadas por mujeres,% listas encabezadas por mujeres,listas competitivas,listas competitivas encabezadas por mujeres,% listas competitivas encabezadas por mujeres,LCEM unión por la patria,LCEM juntos por el cambio,LCEM la libertad avanza,LCEM izquierda,LCEM alianza/partido provincial,camara
0,buenos aires,4,0,0.0,4,0,0.0,NaN,NaN,NaN,NaN,NaN,HCDN
1,caba,4,3,75.0,4,3,75.0,1.0,NaN,1.0,1.0,NaN,HCDN
2,catamarca,3,0,0.0,3,0,0.0,NaN,NaN,NaN,NaN,NaN,HCDN
3,chaco,3,0,0.0,3,0,0.0,NaN,NaN,NaN,NaN,NaN,HCDN
4,chubut,3,0,0.0,3,0,0.0,NaN,NaN,NaN,NaN,NaN,HCDN
5,córdoba,5,3,60.0,4,2,50.0,1.0,NaN,1.0,NaN,NaN,HCDN
6,corrientes,3,1,33.0,3,1,33.0,1.0,NaN,NaN,NaN,NaN,HCDN
7,entre ríos,3,0,0.0,3,0,0.0,NaN,NaN,NaN,NaN,NaN,HCDN
8,formosa,3,1,33.0,3,1,33.0,1.0,NaN,NaN,NaN,NaN,HCDN
9,jujuy,4,1,25.0,4,1,25.0,NaN,NaN,NaN,1.0,NaN,HCDN


#### Encabezamientos 2025

- csv 13. (del año 2025, en donde solo hubo elecciones generales)

In [67]:
E_25_HCDN = dataframes['13. Encabezamientos legislativos nacionales 2025 - Base HCDN']
E_25_HSN = dataframes['13. Encabezamientos legislativos nacionales 2025 - Base HSN']

In [68]:
# Renombrar la columna 'distrito' a 'jurisdiccion y normalizar
E_25_HCDN.rename(columns={'distrito':'jurisdiccion'}, inplace=True)
E_25_HCDN['jurisdiccion'] = E_25_HCDN['jurisdiccion'].apply(normalizar_jurisdiccion)
E_25_HSN.rename(columns={'distrito':'jurisdiccion'}, inplace=True)
E_25_HSN['jurisdiccion'] = E_25_HSN['jurisdiccion'].apply(normalizar_jurisdiccion)

In [69]:
# Quitar el '%' del valor de las columnas '% listas encabezadas por mujeres' y '% listas competitivas encabezadas por mujeres', y convertir a float
E_25_HCDN['% listas encabezadas por mujeres'] = E_25_HCDN['% listas encabezadas por mujeres'].str.replace('%', '').astype(float)
E_25_HCDN['% listas competitivas encabezadas por mujeres'] = E_25_HCDN['% listas competitivas encabezadas por mujeres'].str.replace('%', '').astype(float)
E_25_HSN['% listas encabezadas por mujeres'] = E_25_HSN['% listas encabezadas por mujeres'].str.replace('%', '').astype(float)
E_25_HSN['% listas competitivas encabezadas por mujeres'] = E_25_HSN['% listas competitivas encabezadas por mujeres'].str.replace('%', '').astype(float)

In [70]:
E_25_HSN.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 11 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   jurisdiccion                                   9 non-null      str    
 1   listas                                         9 non-null      int64  
 2   listas encabezadas por mujeres                 9 non-null      int64  
 3   % listas encabezadas por mujeres               9 non-null      float64
 4   listas competitivas                            9 non-null      int64  
 5   listas competitivas encabezadas por mujeres    9 non-null      int64  
 6   % listas competitivas encabezadas por mujeres  9 non-null      float64
 7   LCEM peronismo                                 6 non-null      float64
 8   LCEM la libertad avanza + pro                  4 non-null      float64
 9   LCEM izquierda                                 1 non-null      float6

In [71]:
E_25_HCDN.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 11 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   jurisdiccion                                   25 non-null     str    
 1   listas                                         25 non-null     int64  
 2   listas encabezadas por mujeres                 25 non-null     int64  
 3   % listas encabezadas por mujeres               25 non-null     float64
 4   listas competitivas                            25 non-null     int64  
 5   listas competitivas encabezadas por mujeres    25 non-null     int64  
 6   % listas competitivas encabezadas por mujeres  25 non-null     float64
 7   LCEM peronismo                                 10 non-null     float64
 8   LCEM la libertad avanza + pro                  8 non-null      float64
 9   LCEM izquierda                                 2 non-null      floa

#### ¿Qué hacer con los df creados?

In [72]:
# Guardar todos los df creados en un diccionario
dfs = {
    'CN_base': CN_base,
    'CN_HCDN': CN_HCDN,
    'CN_HSN': CN_HSN,
    'CN_25_27': CN_25_27,
    'E_23_PASO': E_23_PASO,
    'E_23_generales': E_23_generales,
    'E_25_HCDN': E_25_HCDN,
    'E_25_HSN': E_25_HSN
}

In [73]:
# creación del barómetro por eje_ideológico
# detectar los bloques / partidos únicos que existen (y normalizar nombres)

todos_los_bloques = set()

for nombre, df in dfs.items():
    if 'bloque' in df.columns:
        bloques_unicos = df['bloque'].dropna().unique()
        
        for bloque in bloques_unicos:
            bloque_limpio = normalizar_texto(bloque)
            if bloque_limpio:
                todos_los_bloques.add(bloque_limpio)

print("\nLISTA COMPLETA DE BLOQUES:")
for i, bloque in enumerate(sorted(todos_los_bloques), 1):
    print(f"   {i}. {bloque}")


LISTA COMPLETA DE BLOQUES:
   1. accion federal
   2. adelante buenos aires
   3. ahora patria
   4. avanza libertad
   5. avanzar san luis
   6. cambio federal
   7. coalicion civica
   8. coherencia
   9. consenso federal
   10. conviccion federal
   11. cordoba federal
   12. creo
   13. defendamos cordoba
   14. defendamos santa fe
   15. democracia para siempre
   16. despierta chubut
   17. elijo catamarca
   18. encuentro federal
   19. evolucion radical
   20. frente civico
   21. frente civico por santiago
   22. frente civico y social de catamarca
   23. frente de izquierda unidad
   24. frente de izquierda y de los trabajadores
   25. frente de izquierda y de trabajadores - unidad
   26. frente de la concordia misionero
   27. frente de todos
   28. frente nacional y popular
   29. frente pro
   30. frente progresista civico y social
   31. frente renovador de la concordia
   32. fuerza patria
   33. futuro y libertad
   34. hay futuro argentina
   35. identidad bonaerense


In [74]:
# Clasificación ideológica (hay pertidos sin confirmar su ideología)
clasificacion = {
    # Eje 1: Progresista
    'frente de izquierda': 1,
    'partido obrero': 1,
    'mst': 1,
    'izquierda': 1,
    'frente de izquierda unidad': 1,
    'frente de izquierda y de los trabajadores': 1,
    'pts': 1,
    
    # Eje 2: Centro-progresista
    'union por la patria': 2,
    'frente de todos': 2,
    'justicialista': 2,
    'fuerza patria': 2,
    'unidad ciudadana': 2,
    'unidad federal': 2,
    'frente nacional y popular': 2,
    'frente renovador de la concordia': 2,
    
    # Eje 3: Centro
    'coalicion civica': 3,
    'socialista': 3,
    'evolucion radical': 3,
    'encuentro federal': 3,
    'consenso federal': 3,
    'cambio federal': 3,
    'unidos': 3,
    'frente progresista civico y social': 3,
    
    # Eje 4: Centro-conservador
    'juntos por el cambio': 4,
    'pro': 4,
    'union civica radical': 4,
    'ucr': 4,
    'frente pro': 4,
    'mid': 4,
    'republicanos unidos': 4,
    
    # Eje 5: Conservador
    'la libertad avanza': 5,
    'libertad avanza': 5,
    'avanza libertad': 5,
    'futuro y libertad': 5,
    'hay futuro argentina': 5,
    'libertad trabajo y progreso': 5,
}

# Crear barómetro con los partidos clasificados y lista de no confirmados
barometro = []
no_confirmados = []

for bloque in sorted(todos_los_bloques):
    if bloque in clasificacion:
        barometro.append({
            'bloque': bloque,
            'eje': clasificacion[bloque]
        })
    else:
        no_confirmados.append(bloque)

df_barometro = pd.DataFrame(barometro)

In [75]:
df_barometro

,bloque,eje
0,avanza libertad,5
1,cambio federal,3
2,coalicion civica,3
3,consenso federal,3
4,encuentro federal,3
5,evolucion radical,3
6,frente de izquierda unidad,1
7,frente de izquierda y de los trabajadores,1
8,frente de todos,2
9,frente nacional y popular,2


In [76]:
# Agregar barómetro al diccionario dfs
dfs['barometro'] = df_barometro

In [80]:
# Exportar cada cf a csv 

# Crear carpeta para los CSVs (si no existe)
output_folder = 'dataframes_exportados'
os.makedirs(output_folder, exist_ok=True)

# Recorrer el diccionario y exportar cada df
for nombre, df in dfs.items():
    # Limpiar nombre para usarlo como nombre de archivo
    nombre_archivo = f"{nombre}.csv"
    ruta_completa = os.path.join(output_folder, nombre_archivo)
    
    # Exportar a CSV
    df.to_csv(ruta_completa, index=False, encoding='utf-8-sig')
    
print(f"\n Archivos csv guardados en la carpeta: {output_folder}")


 Archivos csv guardados en la carpeta: dataframes_exportados


📂 Explorando archivos CSV...
============================================================

📋 ESTRUCTURA con 8 columnas
   Columnas: ['region', 'jurisdiccion', 'diputadxs', 'cantidad diputadas', 'porcentaje diputadas', 'senadorxs', 'cantidad senadoras', 'porcentaje senadoras']
   Cantidad de archivos: 3
   Ejemplos:
      - 1. Congreso Nacional 2021-2023 - Base.csv
      - 15. Congreso Nacional 2025-2027 - Base.csv
      - 9. Congreso Nacional 2023-2025 - Base.csv

📋 ESTRUCTURA con 7 columnas
   Columnas: ['composicion', 'mandato', 'apellido', 'nombre', 'genero', 'provincia', 'bloque']
   Cantidad de archivos: 2
   Ejemplos:
      - 1. Congreso Nacional 2021-2023 - HCDN.csv
      - 1. Congreso Nacional 2021-2023 - HSN.csv

📋 ESTRUCTURA con 11 columnas
   Columnas: ['distrito', 'listas', 'listas encabezadas por mujeres', '% listas encabezadas por mujeres', 'listas competitivas', 'listas competitivas encabezadas por mujeres', '% listas competitivas encabezadas por mujeres', 'LCEM peronismo', 'LCEM la libertad avanza + pro', 'LCEM izquierda', 'LCEM alianza/partido provincial']
   Cantidad de archivos: 2
   Ejemplos:
      - 13. Encabezamientos legislativos nacionales 2025 - Base HCDN.csv
      - 13. Encabezamientos legislativos nacionales 2025 - Base HSN.csv

📋 ESTRUCTURA con 6 columnas
   Columnas: ['orden', 'lista', 'apellido', 'nombre', 'genero', 'camara']
   Cantidad de archivos: 24
   Ejemplos:
      - 13. Encabezamientos legislativos nacionales 2025 - CABA.csv
      - 13. Encabezamientos legislativos nacionales 2025 - Catamarca.csv
      - 13. Encabezamientos legislativos nacionales 2025 - Chaco.csv

📋 ESTRUCTURA con 7 columnas
   Columnas: ['composicion', 'mandato', 'apellido', 'nombre', 'genero', 'jurisdicción', 'bloque']
   Cantidad de archivos: 4
   Ejemplos:
      - 15. Congreso Nacional 2025-2027 - HCDN.csv
      - 15. Congreso Nacional 2025-2027 - HSN.csv
      - 9. Congreso Nacional 2023-2025 - HCDN.csv

📋 ESTRUCTURA con 9 columnas
   Columnas: ['tipo', 'camara', 'provincia', 'mandato', 'apellido', 'nombre', 'genero', 'seccion', 'bloque']
   Cantidad de archivos: 1
   Ejemplos:
      - 25-27_legislativos - base.csv

📋 ESTRUCTURA con 12 columnas
   Columnas: ['distrito', 'listas', 'listas encabezadas por mujeres', 'listas encabezadas por mujeres sobre el total de listas', 'listas competitivas', 'listas competitivas encabezadas por mujeres', 'porcentaje listas competitivas encabezadas por mujeres sobre el total de listas competitivas', 'listas competitivas encabezadas por mujeres.1', 'LCEM juntos por el cambio', 'LCEM la libertad avanza', 'LCEM izquierda', 'LCEM alianza/partido provincial']
   Cantidad de archivos: 3
   Ejemplos:
      - 5. Encabezamientos legislativos nacionales 2023 - Base generales HCDN.csv
      - 5. Encabezamientos legislativos nacionales 2023 - Base PASO HCDN.csv
      - 5. Encabezamientos legislativos nacionales 2023 - Base PASO HSN.csv

📋 ESTRUCTURA con 12 columnas
   Columnas: ['distrito', 'listas', 'listas encabezadas por mujeres', 'listas encabezadas por mujeres sobre el total de listas', 'listas competitivas', 'listas competitivas encabezadas por mujeres', 'porcentaje listas competitivas encabezadas por mujeres sobre el total de listas competitivas', 'LCEM unión por la patria', 'LCEM la libertad avanza', 'LCEM juntos por el cambio', 'LCEM izquierda', 'LCEM alianza/partido provincial']
   Cantidad de archivos: 1
   Ejemplos:
      - 5. Encabezamientos legislativos nacionales 2023 - Base generales HSN.csv

📋 ESTRUCTURA con 7 columnas
   Columnas: ['orden', 'lista', 'apellido', 'nombre', 'genero', 'camara', 'tipo_de_eleccion']
   Cantidad de archivos: 24
   Ejemplos:
      - 5. Encabezamientos legislativos nacionales 2023 - CABA.csv
      - 5. Encabezamientos legislativos nacionales 2023 - Catamarca.csv
      - 5. Encabezamientos legislativos nacionales 2023 - Chaco.csv